# StoryGuard 로고·배경 이미지 만들기 (교내 GPU · 틸론 주피터)

오픈소스 이미지 생성 모델을 **교내 GPU에서 직접 실행**해 StoryGuard 화면에 쓸 로고와 배경을 만든다.
OpenAI 같은 외부 유료 API는 쓰지 않는다.

**쓰는 법**: 위에서부터 셀을 하나씩 실행한다 (Shift+Enter). 바꿀 곳은 `2. 설정` 셀뿐이다.

| 단계 | 하는 일 | 끝난 기준 |
| --- | --- | --- |
| 1 | GPU 확인 | GPU 이름과 메모리가 보인다 |
| 2 | 설정 | 모델·장수 선택 |
| 3 | 패키지 설치 | 오류 없이 끝남 |
| 4 | 모델 불러오기 | "모델 준비 완료" |
| 5~6 | 로고·배경 후보 생성 | 후보 그림이 격자로 보인다 |
| 7 | 고르기 | 마음에 드는 번호를 적는다 |
| 8 | 화면용으로 다듬어 묶기 | `storyguard_assets.zip` 생김 |
| 9 | GPU 활용 기록 | 제출용 문장이 출력된다 |

**모델과 라이선스** (쓰기 전에 모델 카드에서 다시 확인할 것)
- `flux-schnell`: black-forest-labs/FLUX.1-schnell, Apache-2.0. 4단계로 빠름. bf16 기준 GPU 메모리를 많이 쓴다(약 24GB 이상 권장, 부족하면 CPU 오프로드로 느리게 동작).
- `sdxl`: stabilityai/stable-diffusion-xl-base-1.0, CreativeML Open RAIL++-M. 메모리가 적은 GPU(약 12GB~)에서도 동작.

**주의**
- 그림 안의 글자는 모델이 잘 못 그린다. 그래서 프롬프트에 "글자 없음"을 넣고, `StoryGuard` 글씨는 화면의 Galmuri 글꼴로 쓴다.
- Hugging Face 토큰은 이 두 모델에는 보통 필요 없다. 필요하다고 나오면 4단계 셀 안내대로 입력하고, 노트북에 적어 두지 않는다.

## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
print("PyTorch", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_NAME, GPU_MEM_GB = p.name, round(p.total_memory / 1024**3, 1)
    print("GPU:", GPU_NAME, "| 메모리:", GPU_MEM_GB, "GB")
else:
    GPU_NAME, GPU_MEM_GB = "없음", 0
    print("GPU가 보이지 않습니다. 커널/서버가 GPU용인지 운영진에게 확인하세요.")

## 2. 설정 (여기만 바꾼다)
- `MODEL`: GPU 메모리가 20GB 이상이면 `"flux-schnell"`, 그보다 작으면 `"sdxl"` 추천. `"auto"`면 메모리를 보고 고른다.
- `N_CANDIDATES`: 프롬프트마다 만들 후보 수.
- `SEED_BASE`: 같은 값이면 같은 그림이 다시 나온다(재현용).

In [ ]:
MODEL = "auto"          # "auto" | "flux-schnell" | "sdxl"
N_CANDIDATES = 4
SEED_BASE = 20261105
OUT_DIR = "storyguard_assets"

if MODEL == "auto":
    MODEL = "flux-schnell" if GPU_MEM_GB >= 20 else "sdxl"
MODEL_IDS = {"flux-schnell": "black-forest-labs/FLUX.1-schnell",
             "sdxl": "stabilityai/stable-diffusion-xl-base-1.0"}
print("사용할 모델:", MODEL, "→", MODEL_IDS[MODEL])

## 3. 패키지 설치 (처음 한 번)

In [ ]:
%pip install -q -U diffusers transformers accelerate sentencepiece protobuf safetensors pillow
import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)

## 4. 모델 불러오기
처음에는 모델 파일을 내려받느라 몇 분 걸린다. "401/403"이나 "gated" 오류가 나면 아래 주석의 `login()`을 실행해
Hugging Face 토큰을 **입력창에** 넣는다(노트북에 적지 않음).

In [ ]:
import time, torch
# from huggingface_hub import login; login()   # 토큰이 필요하다고 나올 때만

t0 = time.time()
if MODEL == "flux-schnell":
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(MODEL_IDS[MODEL], torch_dtype=torch.bfloat16)
    STEPS, GUIDANCE = 4, 0.0
else:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_IDS[MODEL], torch_dtype=torch.float16,
                                                     variant="fp16", use_safetensors=True)
    STEPS, GUIDANCE = 30, 6.0

if torch.cuda.is_available() and GPU_MEM_GB >= (20 if MODEL == "flux-schnell" else 10):
    pipe.to("cuda")
    PLACEMENT = "GPU 전체 적재"
else:
    pipe.enable_model_cpu_offload()   # 메모리가 부족하면 느리지만 동작
    PLACEMENT = "CPU 오프로드"
LOAD_SEC = round(time.time() - t0, 1)
print(f"모델 준비 완료 ({PLACEMENT}, {LOAD_SEC}초)")

## 5. 프롬프트
StoryGuard 화면 색: 하늘색 `#DCEBFA`, 남색 `#1B2A4A`, 금색 `#F2B705`, 초록 `#2E9E4F`, 보라 `#7B68C8`. 도트(픽셀) 그림 분위기.
마음에 안 들면 문장을 고쳐 5~6단계를 다시 실행한다.

In [ ]:
STYLE = ("16-bit pixel art, retro game asset, crisp pixels, limited palette of sky blue #DCEBFA, "
         "navy #1B2A4A, gold #F2B705, green #2E9E4F, purple #7B68C8, no text, no letters, no watermark")

PROMPTS = {
    "logo": {
        "prompt": ("app icon of a shield guarding an open storybook with a golden key on top, "
                   "a small green check mark, centered, simple bold shapes, flat light background, " + STYLE),
        "negative": "text, letters, words, signature, blurry, photo, 3d render, cluttered",
        "size": (1024, 1024),
    },
    "background": {
        "prompt": ("wide pixel art landscape of a small village, a forest path and a distant dungeon gate "
                   "under a soft sky blue sky, calm, plenty of empty sky space at the top, low contrast, " + STYLE),
        "negative": "text, letters, characters in foreground, high contrast, busy details, photo, 3d render",
        "size": (1344, 768),
    },
}

## 6. 후보 만들기

In [ ]:
import os, json
from PIL import Image
os.makedirs(OUT_DIR, exist_ok=True)
RECORDS = []

def generate(name):
    spec = PROMPTS[name]
    w, h = spec["size"]
    images = []
    for i in range(N_CANDIDATES):
        seed = SEED_BASE + i
        g = torch.Generator("cpu").manual_seed(seed)
        kw = dict(prompt=spec["prompt"], width=w, height=h, num_inference_steps=STEPS,
                  guidance_scale=GUIDANCE, generator=g)
        if MODEL == "sdxl":
            kw["negative_prompt"] = spec["negative"]
        t = time.time()
        img = pipe(**kw).images[0]
        sec = round(time.time() - t, 1)
        path = f"{OUT_DIR}/{name}_{i}.png"
        img.save(path)
        RECORDS.append({"asset": name, "candidate": i, "seed": seed, "seconds": sec, "path": path,
                        "model": MODEL_IDS[MODEL], "steps": STEPS, "size": [w, h]})
        print(f"{name}_{i}: {sec}초")
        images.append(img)
    return images

def show_grid(images, thumb=320):
    row = Image.new("RGB", (thumb * len(images), thumb * images[0].height // images[0].width), "white")
    for i, im in enumerate(images):
        t = im.copy(); t.thumbnail((thumb, thumb * 4))
        row.paste(t, (i * thumb, 0))
    display(row)

logo_imgs = generate("logo");             show_grid(logo_imgs)
background_imgs = generate("background"); show_grid(background_imgs)

## 7. 고르기 (왼쪽부터 0, 1, 2, ...)

In [ ]:
PICK_LOGO = 0
PICK_BACKGROUND = 0

## 8. 화면용으로 다듬어 묶기
- 로고: 512px, 256px, 파비콘 64px·32px (도트 느낌을 살리려고 NEAREST로 줄임)
- 배경: 1920px 폭 WebP와 PNG
- `credits.json`: 모델·시드·설정·GPU 기록 (발표·제출 근거)

In [ ]:
import zipfile
from datetime import datetime

sel = f"{OUT_DIR}/selected"; os.makedirs(sel, exist_ok=True)
logo = logo_imgs[PICK_LOGO].convert("RGBA")
for px in (512, 256):
    logo.resize((px, px), Image.LANCZOS).save(f"{sel}/logo_{px}.png")
for px in (64, 32):
    logo.resize((px, px), Image.NEAREST).save(f"{sel}/favicon_{px}.png")

bg = background_imgs[PICK_BACKGROUND].convert("RGB")
bg_w = 1920; bg_h = round(bg.height * bg_w / bg.width)
bg_big = bg.resize((bg_w, bg_h), Image.NEAREST)
bg_big.save(f"{sel}/background.webp", quality=85)
bg_big.save(f"{sel}/background.png", optimize=True)

credits = {
    "made_at": datetime.now().isoformat(timespec="seconds"),
    "gpu": GPU_NAME, "gpu_memory_gb": GPU_MEM_GB, "placement": PLACEMENT, "model_load_sec": LOAD_SEC,
    "model": MODEL_IDS[MODEL], "steps": STEPS, "guidance": GUIDANCE,
    "picked": {"logo": PICK_LOGO, "background": PICK_BACKGROUND},
    "prompts": PROMPTS, "candidates": RECORDS,
}
with open(f"{sel}/credits.json", "w", encoding="utf-8") as f:
    json.dump(credits, f, ensure_ascii=False, indent=1)

with zipfile.ZipFile("storyguard_assets.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(sel)):
        z.write(f"{sel}/{fn}", fn)
print("만든 파일:", sorted(os.listdir(sel)))
print("왼쪽 파일 목록에서 storyguard_assets.zip 을 오른쪽 클릭 → Download")

## 9. 교내 GPU 활용 기록 (제출 문장 초안)

In [ ]:
secs = [r["seconds"] for r in RECORDS]
print(f"- 교내 GPU({GPU_NAME}, {GPU_MEM_GB}GB)에서 오픈소스 이미지 생성 모델 {MODEL_IDS[MODEL]}을 직접 실행해 "
      f"서비스 로고와 배경 이미지를 만들었습니다.")
print(f"- 후보 {len(RECORDS)}장 생성, 장당 평균 {sum(secs)/len(secs):.1f}초 ({STEPS}단계, {PLACEMENT}). "
      f"모델·시드·프롬프트는 credits.json에 기록했습니다.")